# 🎓 Notebook 2: Supervised Modeling — Regression & Multi-class Classification

## AuraCart Retail Analytics — MLflow-Tracked Experiments

This notebook implements the supervised learning models for AuraCart's predictive analytics engine:

- **Part A: Regression** — Predict the `price` of incoming orders using Multiple Linear Regression with SGD
- **Part B: Classification** — Classify `delivery_status` (4 classes) and `customer_segment` (3 classes) using Softmax Regression

All experiments are tracked using **MLflow** for reproducibility and comparison.

### Syllabus Alignment
- Module 3: Supervised Learning — Regression (SGD, MSE, MAE, k-fold cross-validation)
- Module 4: Supervised Learning — Classification (Softmax, cross-entropy, confusion matrix)
- Module 6: MLflow experiment tracking

## Step 1: Setup & Load Preprocessed Data

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn models & metrics
from sklearn.linear_model import SGDRegressor, SGDClassifier, LogisticRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    accuracy_score, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

# MLflow
import mlflow
import mlflow.sklearn

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

IMG_DIR = os.path.join('..', 'analysis_images')
ARTIFACTS_DIR = os.path.join('..', 'artifacts')

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [2]:
# Load preprocessed data from Notebook 1
split_data = joblib.load(os.path.join(ARTIFACTS_DIR, 'train_test_split.pkl'))
preprocessor = joblib.load(os.path.join(ARTIFACTS_DIR, 'preprocessor.pkl'))
le_delivery = joblib.load(os.path.join(ARTIFACTS_DIR, 'le_delivery.pkl'))
le_segment = joblib.load(os.path.join(ARTIFACTS_DIR, 'le_segment.pkl'))

X_train = split_data['X_train']
X_test = split_data['X_test']
X_train_processed = split_data['X_train_processed']
X_test_processed = split_data['X_test_processed']
y_price_train = split_data['y_price_train']
y_price_test = split_data['y_price_test']
y_delivery_train_enc = split_data['y_delivery_train_enc']
y_delivery_test_enc = split_data['y_delivery_test_enc']
y_segment_train_enc = split_data['y_segment_train_enc']
y_segment_test_enc = split_data['y_segment_test_enc']
y_segment_train = split_data['y_segment_train']
y_segment_test = split_data['y_segment_test']
y_delivery_train = split_data['y_delivery_train']
y_delivery_test = split_data['y_delivery_test']
feature_names = split_data['feature_names']

print(f'Training set: {X_train_processed.shape}')
print(f'Testing set: {X_test_processed.shape}')
print(f'\nDelivery Status classes: {le_delivery.classes_}')
print(f'Customer Segment classes: {le_segment.classes_}')

Training set: (8000, 23)
Testing set: (2000, 23)

Delivery Status classes: ['Delivered' 'Pending' 'Returned' 'Shipped']
Customer Segment classes: ['New' 'Returning' 'VIP']


In [3]:
# Setup MLflow
mlflow.set_tracking_uri('mlruns')
print('MLflow tracking URI set.')

MLflow tracking URI set.


---
# Part A: Continuous Price Prediction (Regression)

## Task 3.2: Multiple Linear Regression with Stochastic Gradient Descent

We implement a Multiple Linear Regression model using `SGDRegressor` to predict the continuous `price` value of AuraCart orders. SGD (Stochastic Gradient Descent) iteratively optimizes the model weights by computing gradients on mini-batches of data.

### Why SGD for Regression?
- Scales well to large datasets (processes samples incrementally)
- Allows fine-grained control over learning rate, epochs, and convergence
- Directly applicable to the gradient descent concepts from Module 3

In [4]:
# Experiment: Train SGDRegressor with different hyperparameters
mlflow.set_experiment('AuraCart_Price_Regression')

# Hyperparameter configurations to test
regression_configs = [
    {'learning_rate': 'constant', 'eta0': 0.001, 'max_iter': 5000, 'tol': 1e-4},
    {'learning_rate': 'constant', 'eta0': 0.01, 'max_iter': 5000, 'tol': 1e-4},
    {'learning_rate': 'invscaling', 'eta0': 0.01, 'max_iter': 5000, 'tol': 1e-4},
    {'learning_rate': 'adaptive', 'eta0': 0.01, 'max_iter': 10000, 'tol': 1e-4},
    {'learning_rate': 'optimal', 'eta0': 0.01, 'max_iter': 10000, 'tol': 1e-5},
]

best_regression_mse = float('inf')
best_regression_model = None

for i, config in enumerate(regression_configs):
    with mlflow.start_run(run_name=f'SGD_Regression_Run_{i+1}'):
        # Log parameters
        mlflow.log_params(config)
        
        # Train model
        sgd_reg = SGDRegressor(
            loss='squared_error',
            penalty='l2',
            random_state=42,
            **config
        )
        sgd_reg.fit(X_train_processed, y_price_train)
        
        # Predict
        y_pred = sgd_reg.predict(X_test_processed)
        
        # Evaluate
        mse = mean_squared_error(y_price_test, y_pred)
        mae = mean_absolute_error(y_price_test, y_pred)
        rmse = np.sqrt(mse)
        
        # Log metrics
        mlflow.log_metric('MSE', mse)
        mlflow.log_metric('MAE', mae)
        mlflow.log_metric('RMSE', rmse)
        
        # Log model
        mlflow.sklearn.log_model(sgd_reg, 'model')
        
        print(f'Run {i+1}: lr={config["learning_rate"]}, eta0={config["eta0"]}, '
              f'max_iter={config["max_iter"]} => MSE={mse:.4f}, MAE={mae:.4f}, RMSE={rmse:.4f}')
        
        if mse < best_regression_mse:
            best_regression_mse = mse
            best_regression_model = sgd_reg

print(f'\n\u2705 Best Regression MSE: {best_regression_mse:.4f}')

2026/04/04 13:09:41 INFO mlflow.tracking.fluent: Experiment with name 'AuraCart_Price_Regression' does not exist. Creating a new experiment.


2026/04/04 13:09:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/04 13:09:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/04/04 13:09:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run 1: lr=constant, eta0=0.001, max_iter=5000 => MSE=20656.1768, MAE=123.8972, RMSE=143.7226


...

2026/04/04 13:10:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 4: lr=adaptive, eta0=0.01, max_iter=10000 => MSE=20649.4826, MAE=123.9408, RMSE=143.6993


2026/04/04 13:10:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/04 13:10:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 5: lr=optimal, eta0=0.01, max_iter=10000 => MSE=6878592332.8885, MAE=82911.0355, RMSE=82937.2795

✅ Best Regression MSE: 20649.4826


### Understanding the Gradient Descent Parameters

- **Learning Rate (`eta0`)**: Controls how large each update step is during training. A high learning rate may cause the model to overshoot the optimal solution, while a low rate makes training slow.
- **Max Iterations (`max_iter`)**: The number of passes over the training dataset. More iterations allow the model to converge further but risk overfitting.
- **Tolerance (`tol`)**: The stopping criterion. Training stops when the improvement is less than this value.
- **Learning Rate Schedule**: `constant` keeps eta0 fixed; `invscaling` decreases it over time; `adaptive` reduces it when training stalls; `optimal` uses a theoretically optimal rate.

### Regression Model Evaluation

**MSE vs MAE:**
- **MSE** (Mean Squared Error): Penalizes large errors heavily due to squaring. Useful when large deviations are particularly costly.
- **MAE** (Mean Absolute Error): Treats all errors equally. More robust to outliers.

**Business Impact:** For AuraCart's pricing system, large prediction errors can lead to significant revenue loss (overpricing loses customers, underpricing loses profit). MSE is thus the more business-relevant metric.

In [5]:
# Detailed evaluation of best regression model
y_pred_best = best_regression_model.predict(X_test_processed)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Actual vs Predicted
axes[0].scatter(y_price_test, y_pred_best, alpha=0.3, color='steelblue', s=10)
axes[0].plot([y_price_test.min(), y_price_test.max()],
             [y_price_test.min(), y_price_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title('Actual vs Predicted Price')

# Residuals
residuals = y_price_test - y_pred_best
axes[1].hist(residuals, bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')
axes[1].axvline(x=0, color='red', linestyle='--', lw=2)

plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'regression_evaluation.png'), dpi=150, bbox_inches='tight')
plt.show()

In [6]:
# K-Fold Cross-Validation for reliable evaluation
print('--- K-Fold Cross-Validation (k=5) ---')
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_mse_scores = cross_val_score(best_regression_model, X_train_processed, y_price_train,
                                 cv=kf, scoring='neg_mean_squared_error')
cv_mae_scores = cross_val_score(best_regression_model, X_train_processed, y_price_train,
                                 cv=kf, scoring='neg_mean_absolute_error')

print(f'MSE per fold: {-cv_mse_scores}')
print(f'Mean MSE: {-cv_mse_scores.mean():.4f} (+/- {cv_mse_scores.std():.4f})')
print(f'\nMAE per fold: {-cv_mae_scores}')
print(f'Mean MAE: {-cv_mae_scores.mean():.4f} (+/- {cv_mae_scores.std():.4f})')

# Analysis of bias/variance
train_mse = mean_squared_error(y_price_train, best_regression_model.predict(X_train_processed))
test_mse = mean_squared_error(y_price_test, y_pred_best)
print(f'\nTrain MSE: {train_mse:.4f}')
print(f'Test MSE: {test_mse:.4f}')
if train_mse < test_mse * 0.5:
    print('\u26a0\ufe0f Possible OVERFITTING: Train error much lower than test error.')
elif train_mse > test_mse * 1.5:
    print('\u26a0\ufe0f Possible UNDERFITTING: Both train and test errors are high.')
else:
    print('\u2705 Model shows reasonable bias-variance tradeoff.')

--- K-Fold Cross-Validation (k=5) ---


MSE per fold: [20407.88523469 19253.31606725 20002.12664467 20003.40376642
 19857.95513749]
Mean MSE: 19904.9374 (+/- 373.8758)

MAE per fold: [123.58235928 119.6312803  121.73583973 122.37283671 121.28399738]
Mean MAE: 121.7213 (+/- 1.2998)

Train MSE: 19794.3456
Test MSE: 20649.4826
✅ Model shows reasonable bias-variance tradeoff.


---
# Part B: Multi-class Classification

## Task 3.3: Softmax Regression (Multinomial Logistic Regression)

### Why not Linear Regression for Classification?
Linear regression predicts continuous values on an unbounded scale. For categorical outcomes:
- Predictions can fall outside [0, 1] and have no probabilistic interpretation
- The assumption of linear relationship between features and output doesn't apply to discrete classes
- Class boundaries require non-linear decision functions

### How Softmax Works
The Softmax function converts raw model outputs (logits) into probabilities across K classes:
$$P(y=k|x) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

We use **log-loss (categorical cross-entropy)** during training, which measures the difference between predicted probabilities and true class labels.

**Note:** In scikit-learn 1.8+, LogisticRegression automatically handles multi-class with the multinomial/softmax approach when there are >2 classes.

In [7]:
# --- Classification Task 1: Delivery Status ---
mlflow.set_experiment('AuraCart_Delivery_Classification')

classification_configs = [
    {'solver': 'saga', 'C': 1.0, 'max_iter': 5000, 'class_weight': None},
    {'solver': 'saga', 'C': 1.0, 'max_iter': 5000, 'class_weight': 'balanced'},
    {'solver': 'lbfgs', 'C': 0.1, 'max_iter': 5000, 'class_weight': 'balanced'},
    {'solver': 'lbfgs', 'C': 10.0, 'max_iter': 5000, 'class_weight': 'balanced'},
]

best_delivery_f1 = 0
best_delivery_model = None

for i, config in enumerate(classification_configs):
    with mlflow.start_run(run_name=f'Delivery_Classification_Run_{i+1}'):
        mlflow.log_params(config)
        
        # LogisticRegression automatically uses softmax (multinomial) for >2 classes
        model = LogisticRegression(
            random_state=42,
            **config
        )
        model.fit(X_train_processed, y_delivery_train_enc)
        
        y_pred = model.predict(X_test_processed)
        
        acc = accuracy_score(y_delivery_test_enc, y_pred)
        f1_macro = f1_score(y_delivery_test_enc, y_pred, average='macro')
        f1_weighted = f1_score(y_delivery_test_enc, y_pred, average='weighted')
        
        mlflow.log_metric('accuracy', acc)
        mlflow.log_metric('f1_macro', f1_macro)
        mlflow.log_metric('f1_weighted', f1_weighted)
        mlflow.sklearn.log_model(model, 'model')
        
        print(f'Run {i+1}: solver={config["solver"]}, C={config["C"]}, '
              f'class_weight={config["class_weight"]} => Acc={acc:.4f}, F1_macro={f1_macro:.4f}')
        
        if f1_macro > best_delivery_f1:
            best_delivery_f1 = f1_macro
            best_delivery_model = model

print(f'\n\u2705 Best Delivery F1 (macro): {best_delivery_f1:.4f}')

2026/04/04 13:10:35 INFO mlflow.tracking.fluent: Experiment with name 'AuraCart_Delivery_Classification' does not exist. Creating a new experiment.


2026/04/04 13:10:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/04 13:10:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 1: solver=saga, C=1.0, class_weight=None => Acc=0.7115, F1_macro=0.2079


2026/04/04 13:10:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

2026/04/04 13:11:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 3: solver=lbfgs, C=0.1, class_weight=balanced => Acc=0.1790, F1_macro=0.1531


2026/04/04 13:11:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/04 13:11:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 4: solver=lbfgs, C=10.0, class_weight=balanced => Acc=0.1770, F1_macro=0.1516

✅ Best Delivery F1 (macro): 0.2079


In [8]:
# --- Classification Task 2: Customer Segment (THIS MODEL WILL BE DEPLOYED) ---
mlflow.set_experiment('AuraCart_Segment_Classification')

segment_configs = [
    {'solver': 'saga', 'C': 1.0, 'max_iter': 5000, 'class_weight': None},
    {'solver': 'saga', 'C': 1.0, 'max_iter': 5000, 'class_weight': 'balanced'},
    {'solver': 'lbfgs', 'C': 0.1, 'max_iter': 5000, 'class_weight': 'balanced'},
    {'solver': 'lbfgs', 'C': 1.0, 'max_iter': 5000, 'class_weight': 'balanced'},
    {'solver': 'lbfgs', 'C': 10.0, 'max_iter': 5000, 'class_weight': 'balanced'},
]

best_segment_f1 = 0
best_segment_model = None
best_segment_run_params = None

for i, config in enumerate(segment_configs):
    with mlflow.start_run(run_name=f'Segment_Classification_Run_{i+1}'):
        mlflow.log_params(config)
        
        # LogisticRegression automatically uses softmax (multinomial) for >2 classes
        model = LogisticRegression(
            random_state=42,
            **config
        )
        model.fit(X_train_processed, y_segment_train_enc)
        
        y_pred = model.predict(X_test_processed)
        
        acc = accuracy_score(y_segment_test_enc, y_pred)
        f1_macro = f1_score(y_segment_test_enc, y_pred, average='macro')
        f1_weighted = f1_score(y_segment_test_enc, y_pred, average='weighted')
        
        mlflow.log_metric('accuracy', acc)
        mlflow.log_metric('f1_macro', f1_macro)
        mlflow.log_metric('f1_weighted', f1_weighted)
        mlflow.sklearn.log_model(model, 'model')
        
        print(f'Run {i+1}: solver={config["solver"]}, C={config["C"]}, '
              f'class_weight={config["class_weight"]} => Acc={acc:.4f}, F1_macro={f1_macro:.4f}')
        
        if f1_macro > best_segment_f1:
            best_segment_f1 = f1_macro
            best_segment_model = model
            best_segment_run_params = config

print(f'\n\u2705 Best Segment F1 (macro): {best_segment_f1:.4f}')
print(f'Best Config: {best_segment_run_params}')

2026/04/04 13:11:20 INFO mlflow.tracking.fluent: Experiment with name 'AuraCart_Segment_Classification' does not exist. Creating a new experiment.


2026/04/04 13:11:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/04 13:11:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 1: solver=saga, C=1.0, class_weight=None => Acc=0.4850, F1_macro=0.2198


2026/04/04 13:11:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

2026/04/04 13:11:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 4: solver=lbfgs, C=1.0, class_weight=balanced => Acc=0.2580, F1_macro=0.2358


2026/04/04 13:12:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/04 13:12:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 5: solver=lbfgs, C=10.0, class_weight=balanced => Acc=0.2580, F1_macro=0.2358

✅ Best Segment F1 (macro): 0.2365
Best Config: {'solver': 'saga', 'C': 1.0, 'max_iter': 5000, 'class_weight': 'balanced'}


## Task 3.4: Performance Evaluation & Risk Analysis

### Confusion Matrix & Class-wise Metrics
Overall accuracy can be misleading with imbalanced data. We examine per-class precision, recall, and F1-score to understand performance on each class individually.

In [9]:
# --- Delivery Status: Detailed Evaluation ---
y_delivery_pred = best_delivery_model.predict(X_test_processed)

print('=== Delivery Status Classification Report ===')
print(classification_report(y_delivery_test_enc, y_delivery_pred,
                            target_names=le_delivery.classes_))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_delivery_test_enc, y_delivery_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le_delivery.classes_)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix: Delivery Status')
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'confusion_matrix_delivery.png'), dpi=150, bbox_inches='tight')
plt.show()

=== Delivery Status Classification Report ===
              precision    recall  f1-score   support

   Delivered       0.71      1.00      0.83      1423
     Pending       0.00      0.00      0.00        88
    Returned       0.00      0.00      0.00       110
     Shipped       0.00      0.00      0.00       379

    accuracy                           0.71      2000
   macro avg       0.18      0.25      0.21      2000
weighted avg       0.51      0.71      0.59      2000



In [10]:
# --- Customer Segment: Detailed Evaluation ---
y_segment_pred = best_segment_model.predict(X_test_processed)

print('=== Customer Segment Classification Report ===')
print(classification_report(y_segment_test_enc, y_segment_pred,
                            target_names=le_segment.classes_))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm_seg = confusion_matrix(y_segment_test_enc, y_segment_pred)
disp_seg = ConfusionMatrixDisplay(cm_seg, display_labels=le_segment.classes_)
disp_seg.plot(ax=ax, cmap='Purples', values_format='d')
ax.set_title('Confusion Matrix: Customer Segment')
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'confusion_matrix_segment.png'), dpi=150, bbox_inches='tight')
plt.show()

=== Customer Segment Classification Report ===
              precision    recall  f1-score   support

         New       0.04      0.34      0.07       108
   Returning       0.45      0.32      0.37       921
         VIP       0.42      0.19      0.26       971

    accuracy                           0.26      2000
   macro avg       0.30      0.28      0.24      2000
weighted avg       0.41      0.26      0.30      2000



### Decision Threshold Analysis

Instead of always choosing the class with the highest probability, we can adjust decision thresholds to improve detection of rare but important classes. For example, correctly identifying a "Returned" order is critical for AuraCart's loss-prevention strategy.

In [11]:
# Predicted probabilities from the delivery status model
y_delivery_proba = best_delivery_model.predict_proba(X_test_processed)

print('Softmax probability output shape:', y_delivery_proba.shape)
print('\nExample predictions (first 10 samples):')
prob_df = pd.DataFrame(y_delivery_proba, columns=le_delivery.classes_)
prob_df['Predicted'] = le_delivery.inverse_transform(y_delivery_pred)
prob_df['Actual'] = y_delivery_test.values
print(prob_df.head(10).to_string())

print('\nNote: The Softmax function ensures all class probabilities sum to 1.0')
print('Adjusting thresholds for minority classes (e.g., lowering threshold for "Returned")')
print('can increase recall for that class, at the cost of precision.')

Softmax probability output shape: (2000, 4)

Example predictions (first 10 samples):
   Delivered   Pending  Returned   Shipped  Predicted     Actual
0   0.717453  0.058903  0.041488  0.182156  Delivered  Delivered
1   0.696583  0.065302  0.053370  0.184745  Delivered  Delivered
2   0.701687  0.065857  0.039527  0.192929  Delivered    Shipped
3   0.741592  0.049935  0.045095  0.163378  Delivered  Delivered
4   0.713732  0.053867  0.049759  0.182643  Delivered  Delivered
5   0.718180  0.043612  0.064056  0.174152  Delivered  Delivered
6   0.695058  0.057478  0.047173  0.200292  Delivered  Delivered
7   0.738621  0.071082  0.052167  0.138130  Delivered    Shipped
8   0.727431  0.042233  0.059256  0.171080  Delivered    Shipped
9   0.688990  0.053404  0.047026  0.210580  Delivered  Delivered

Note: The Softmax function ensures all class probabilities sum to 1.0
Adjusting thresholds for minority classes (e.g., lowering threshold for "Returned")
can increase recall for that class, at the co

### Precision-Recall Trade-offs & Asymmetric Risk Analysis

In AuraCart's business context:

- **False Positive (predicting "Returned" when it won't be):** AuraCart may unnecessarily process a return preparation, wasting resources. Moderate cost.
- **False Negative (failing to predict a return):** AuraCart misses an at-risk order, leading to customer dissatisfaction, increased return costs, and potential churn. HIGH cost.

**Conclusion:** For the "Returned" class, **recall is more important than precision**, because the cost of missing a return (false negative) is higher than the cost of a false alarm (false positive).

Similarly, for customer segments, misclassifying a VIP customer as "New" means they miss personalized attention, leading to churn of high-value clients.

In [12]:
# Save the best classification model for deployment (customer_segment)
joblib.dump(best_segment_model, os.path.join(ARTIFACTS_DIR, 'best_segment_classifier.pkl'))
joblib.dump(best_delivery_model, os.path.join(ARTIFACTS_DIR, 'best_delivery_classifier.pkl'))
joblib.dump(best_regression_model, os.path.join(ARTIFACTS_DIR, 'best_regression_model.pkl'))

print('\u2705 Best models saved to artifacts directory.')
print(f'  - best_segment_classifier.pkl (for Vertex AI deployment)')
print(f'  - best_delivery_classifier.pkl')
print(f'  - best_regression_model.pkl')

✅ Best models saved to artifacts directory.
  - best_segment_classifier.pkl (for Vertex AI deployment)
  - best_delivery_classifier.pkl
  - best_regression_model.pkl


## Summary

In this notebook, we have:

1. **Regression (Price Prediction):** Trained multiple SGDRegressor models with different hyperparameters, evaluated using MSE/MAE, and performed 5-fold cross-validation
2. **Classification (Delivery Status):** Trained Softmax regression with class imbalance handling, analyzed confusion matrix and per-class metrics
3. **Classification (Customer Segment):** Trained the champion model for Vertex AI deployment with MLflow tracking
4. **Decision Threshold Analysis:** Explored probability-based predictions and threshold calibration
5. **Risk Analysis:** Discussed asymmetric costs of false positives vs false negatives in AuraCart's context
6. **MLflow Tracking:** All experiments systematically tracked with parameters, metrics, and model artifacts